# Step 8 & 9: CLI Reporting Tool Demo and Edge Case Handling
The actual tool lives in `../scripts/report_cli.py`. This notebook runs it and tests edge cases.

In [1]:
import subprocess

def run_cli(args):
    result = subprocess.run(
        ['python3', '../scripts/report_cli.py'] + args,
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)


## Revenue report

In [2]:
run_cli(['--report', 'revenue'])

+---------+-----------+
| month   |   revenue |
+=========+===========+
| 2024-07 |   36050.4 |
+---------+-----------+
| 2024-08 |   49655.6 |
+---------+-----------+
| 2024-09 |   55827.2 |
+---------+-----------+
| 2024-10 |   57706.6 |
+---------+-----------+
| 2024-11 |   49379.8 |
+---------+-----------+
| 2024-12 |   56602.1 |
+---------+-----------+
| 2025-01 |   57879.4 |
+---------+-----------+
| 2025-02 |   46339.1 |
+---------+-----------+
| 2025-03 |   73189.3 |
+---------+-----------+
| 2025-04 |   62680.1 |
+---------+-----------+
| 2025-05 |   63257.5 |
+---------+-----------+
| 2025-06 |   62274.6 |
+---------+-----------+
| 2025-07 |   69645.4 |
+---------+-----------+
| 2025-08 |   95647.7 |
+---------+-----------+
| 2025-09 |   51726.8 |
+---------+-----------+
| 2025-10 |   70929.2 |
+---------+-----------+
| 2025-11 |   76157.6 |
+---------+-----------+
| 2025-12 |   64076   |
+---------+-----------+
| 2026-01 |   59719.8 |
+---------+-----------+
| 2026-02 |   61

## Top customers report

In [3]:
run_cli(['--report', 'top_customers'])

+---------------+-----------------+-----------+
|   customer_id | name            |   revenue |
+===============+=================+===========+
|           174 | Sandra Davis    |   22741.6 |
+---------------+-----------------+-----------+
|            89 | Mary Peck       |   21058   |
+---------------+-----------------+-----------+
|            80 | Pamela Lopez    |   18470   |
+---------------+-----------------+-----------+
|           200 | Patricia Morrow |   17466.5 |
+---------------+-----------------+-----------+
|           120 | Michael Craig   |   17350.2 |
+---------------+-----------------+-----------+
|            71 | Jeffrey Wood    |   17164.8 |
+---------------+-----------------+-----------+
|            90 | Charles Brown   |   16929.3 |
+---------------+-----------------+-----------+
|            70 | Lori Garcia     |   16882   |
+---------------+-----------------+-----------+
|           161 | Robert Monroe   |   16481.3 |
+---------------+-----------------+-----

## Top products report

In [4]:
run_cli(['--report', 'top_products'])

+----------------+--------------+-----------+
| product_name   |   units_sold |   revenue |
+================+==============+===========+
| Ok Lite        |          113 |   37706.3 |
+----------------+--------------+-----------+
| Morning Lite   |          127 |   34721.3 |
+----------------+--------------+-----------+
| Put            |          136 |   34400.3 |
+----------------+--------------+-----------+
| Floor Pro      |          128 |   33742.1 |
+----------------+--------------+-----------+
| Car Max        |          122 |   33686.6 |
+----------------+--------------+-----------+
| Push Pro       |          107 |   33413   |
+----------------+--------------+-----------+
| Them           |          131 |   33102.6 |
+----------------+--------------+-----------+
| Yeah Pro       |          115 |   32311.4 |
+----------------+--------------+-----------+
| Prove          |          117 |   30691.9 |
+----------------+--------------+-----------+
| None Max       |          122 | 

## Retention report

In [5]:
run_cli(['--report', 'retention'])

+----------------+--------------------+---------------+
| cohort_month   |   active_customers | order_month   |
+================+====================+===============+
| 2024-07        |                 19 | 2024-07       |
+----------------+--------------------+---------------+
| 2024-07        |                  3 | 2024-08       |
+----------------+--------------------+---------------+
| 2024-07        |                  3 | 2024-09       |
+----------------+--------------------+---------------+
| 2024-07        |                  3 | 2024-11       |
+----------------+--------------------+---------------+
| 2024-07        |                  3 | 2024-12       |
+----------------+--------------------+---------------+
| 2024-07        |                  4 | 2025-01       |
+----------------+--------------------+---------------+
| 2024-07        |                  3 | 2025-02       |
+----------------+--------------------+---------------+
| 2024-07        |                  3 | 2025-03 

## Segments report

In [6]:
run_cli(['--report', 'segments'])

+-----------+---------------+-----------+
| segment   |   n_customers |   revenue |
+===========+===============+===========+
| New       |            66 |    524524 |
+-----------+---------------+-----------+
| Premium   |            51 |    400730 |
+-----------+---------------+-----------+
| Regular   |            67 |    524751 |
+-----------+---------------+-----------+



## Edge case: invalid report name

In [7]:
run_cli(['--report', 'bogus'])


STDERR: usage: report_cli.py [-h] --report
                     {revenue,top_customers,top_products,retention,segments}
report_cli.py: error: argument --report: invalid choice: 'bogus' (choose from 'revenue', 'top_customers', 'top_products', 'retention', 'segments')



## Edge case: missing required argument

In [8]:
run_cli([])


STDERR: usage: report_cli.py [-h] --report
                     {revenue,top_customers,top_products,retention,segments}
report_cli.py: error: the following arguments are required: --report



## Edge case: empty result set
Query a report scoped to a future date range that has no matching rows.

In [9]:
import sqlite3

conn = sqlite3.connect('../data/ecommerce.db')
cur = conn.cursor()
cur.execute("SELECT * FROM orders WHERE order_date > '2099-01-01'")
rows = cur.fetchall()

if not rows:
    print('No data found for this date range.')
else:
    print(rows)

conn.close()


No data found for this date range.


## Edge case: single customer with zero orders

In [10]:
conn = sqlite3.connect('../data/ecommerce.db')
cur = conn.cursor()

cur.execute("""
SELECT c.customer_id, c.name, COUNT(o.order_id) AS n_orders
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
HAVING n_orders = 0
LIMIT 5
""")
zero_order_customers = cur.fetchall()
print('customers with zero orders:', zero_order_customers)

conn.close()


customers with zero orders: [(134, 'Victor Brown', 0)]


## Edge case: database connection error
Point at a path that does not exist and confirm it fails gracefully instead of crashing.

In [11]:
import sys
sys.path.insert(0, '../scripts')
import report_cli

report_cli.DB_PATH = 'does/not/exist.db'
try:
    report_cli.get_connection()
except SystemExit:
    print('handled gracefully: exited instead of crashing')


Database connection error: unable to open database file
handled gracefully: exited instead of crashing


## Save sample outputs to output/sample_reports/

In [12]:
import subprocess

reports = ['revenue', 'top_customers', 'top_products', 'retention', 'segments']

for r in reports:
    result = subprocess.run(
        ['python3', '../scripts/report_cli.py', '--report', r],
        capture_output=True, text=True
    )
    with open(f'../output/sample_reports/{r}.txt', 'w') as f:
        f.write(result.stdout)

print('sample reports saved')


sample reports saved
